# Meal Plan Agent

A LangGraph agent that:
1. Generates a weekly meal plan for a family
2. Sends an email with a link to a chat interface to refine the plan

In [ ]:
%pip install langgraph langchain-openai langchain-core

In [ ]:
import os
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages

In [ ]:
# Configure your OpenAI API key
# os.environ["OPENAI_API_KEY"] = "sk-..."

## Define the family profile and state

In [ ]:
FAMILY_PROFILE = {
    "name": "The Johnsons",
    "members": [
        {"name": "Mark", "age": 42, "preferences": "likes grilling, no shellfish allergy"},
        {"name": "Sarah", "age": 39, "preferences": "vegetarian-leaning, loves Mediterranean food"},
        {"name": "Lily", "age": 12, "preferences": "picky eater, likes pasta and chicken"},
        {"name": "Ethan", "age": 8, "preferences": "loves tacos and pizza, won't eat mushrooms"},
    ],
    "budget": "moderate",
    "meals_per_day": 3,
    "days": 7,
}


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    meal_plan: str
    email_sent: bool
    refinement_url: str

## Define agent nodes

In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0.7)


def generate_meal_plan(state: AgentState) -> AgentState:
    """Generate a weekly meal plan tailored to the family."""
    family = FAMILY_PROFILE
    members_desc = "\n".join(
        f"  - {m['name']} (age {m['age']}): {m['preferences']}"
        for m in family["members"]
    )

    prompt = f"""Create a {family['days']}-day meal plan ({family['meals_per_day']} meals/day) \
for {family['name']}.

Family members:
{members_desc}

Budget: {family['budget']}

Requirements:
- Balance everyone's preferences across the week
- Include a grocery list at the end
- Keep meals practical for a busy family
- Format clearly with days and meal types (Breakfast, Lunch, Dinner)"""

    response = llm.invoke([
        SystemMessage(content="You are a family meal planning assistant."),
        HumanMessage(content=prompt),
    ])

    return {
        "messages": [AIMessage(content=f"Meal plan generated for {family['name']}.")],
        "meal_plan": response.content,
        "email_sent": False,
    }

In [ ]:
def send_refinement_email(state: AgentState) -> AgentState:
    """Send an email with the meal plan and a link to refine it via chat."""
    family = FAMILY_PROFILE

    # In production this would be a real URL to your chat interface
    # (e.g. a LangGraph Studio deployment or custom web app).
    refinement_url = "https://planyourmeals.app/chat/session/abc123"

    email_body = f"""Hi {family['name']},

Your weekly meal plan is ready! Here's a preview:

{state['meal_plan'][:500]}...

---
Want to make changes? Swap a meal, adjust portions, or accommodate a last-minute
guest — just click the link below to chat with our meal planning assistant:

  {refinement_url}

Happy cooking!
— PlanYourMeals"""

    # --- Stub: replace with real email sending (e.g. SendGrid, SES, SMTP) ---
    recipient = "sarah.johnson@example.com"
    print(f"=== EMAIL TO: {recipient} ===")
    print(f"Subject: Your Weekly Meal Plan is Ready!")
    print(f"---")
    print(email_body)
    print("=== END EMAIL ===")

    return {
        "messages": [AIMessage(content=f"Email sent to {recipient} with refinement link.")],
        "email_sent": True,
        "refinement_url": refinement_url,
    }

## Build the graph

In [ ]:
graph_builder = StateGraph(AgentState)

graph_builder.add_node("generate_meal_plan", generate_meal_plan)
graph_builder.add_node("send_refinement_email", send_refinement_email)

graph_builder.set_entry_point("generate_meal_plan")
graph_builder.add_edge("generate_meal_plan", "send_refinement_email")
graph_builder.add_edge("send_refinement_email", END)

agent = graph_builder.compile()

# Visualize the graph
try:
    from IPython.display import Image, display
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print(agent.get_graph().draw_ascii())

## Run the agent

In [ ]:
result = agent.invoke({
    "messages": [HumanMessage(content="Please create this week's meal plan.")],
    "meal_plan": "",
    "email_sent": False,
    "refinement_url": "",
})

print("\n" + "=" * 60)
print("FULL MEAL PLAN")
print("=" * 60)
print(result["meal_plan"])
print(f"\nEmail sent: {result['email_sent']}")
print(f"Refinement URL: {result['refinement_url']}")